In [1]:
import sys
sys.path.append('../src')

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torchvision.models import vgg19
from skimage.color import lab2rgb
from pathlib import Path
import time
import os

#noinspection PyUnresolvedReferences
from model_resnet_unet_lab import ResNetUNet
#noinspection PyUnresolvedReferences
from dataset_lab import get_dataloader

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Устройство: {device}")

Устройство: cpu


In [ ]:
class VisualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = vgg19(weights='IMAGENET1K_V1').features
        self.slice = nn.Sequential(*list(vgg.children())[:18]).eval()
        for param in self.slice.parameters():
            param.requires_grad = False

    def forward(self, fake_ab, real_ab, l_batch):
        # Чтобы VGG мог оценить картинку, нам нужно собрать Lab в некое подобие RGB
        # Для скорости просто склеиваем L и ab
        fake_img = torch.cat([l_batch, fake_ab], dim=1)
        real_img = torch.cat([l_batch, real_ab], dim=1)
        return nn.functional.l1_loss(self.slice(fake_img), self.slice(real_img))

criterion_l1 = nn.L1Loss()
criterion_perceptual = VisualLoss().to(device)

Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to C:\Users\Denis/.cache\torch\hub\checkpoints\vgg19-dcbb9e9d.pth


  2%|▏         | 11.4M/548M [01:00<1:16:27, 123kB/s]

In [ ]:
model = ResNetUNet(n_class=2).to(device)
optimizer = optim.Adam(model.parameters(), lr=2e-4, betas=(0.5, 0.999))

In [ ]:
def lab_to_rgb(l_tensor, ab_tensor):
    """Преобразование тензоров обратно в читаемый RGB"""
    l = (l_tensor.detach().cpu().numpy() + 1.0) * 50.0
    ab = ab_tensor.detach().cpu().numpy() * 128.0
    lab = np.concatenate([l, ab], axis=0).transpose(1, 2, 0)
    rgb = lab2rgb(lab.astype(np.float64))
    return (rgb * 255).astype(np.uint8)

def visualize_results(l_batch, ab_batch, model, epoch):
    model.eval()
    with torch.no_grad():
        pred_ab = model(l_batch.to(device))

    l_single = l_batch[0]
    ab_single = ab_batch[0]
    pred_single = pred_ab[0]

    res_img = lab_to_rgb(l_single, pred_single)
    gt_img = lab_to_rgb(l_single, ab_single)

    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(res_img)
    plt.title(f'Predicted (Epoch {epoch})')
    plt.subplot(1, 2, 2)
    plt.imshow(gt_img)
    plt.title('Ground Truth')
    plt.show()

In [ ]:
epochs = 50
dataloader = get_dataloader('../data/processed/test', batch_size=32)

for epoch in range(epochs):
    model.train()
    epoch_loss = 0

    for i, (l_batch, ab_batch) in enumerate(dataloader):
        l_batch, ab_batch = l_batch.to(device), ab_batch.to(device)

        optimizer.zero_grad()

        # Предсказание
        pred_ab = model(l_batch)

        # Расчет лоссов
        loss_l1 = criterion_l1(pred_ab, ab_batch)
        loss_vgg = criterion_perceptual(pred_ab, ab_batch, l_batch)

        # Итоговый лосс (баланс 1:0.1 обычно работает хорошо)
        loss = loss_l1 + 0.1 * loss_vgg

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss/len(dataloader):.4f}")

    # Визуализация каждые 5 эпох
    if (epoch + 1) % 5 == 0:
        visualize_results(l_batch, ab_batch, model, epoch + 1)